In [1]:
from pathlib import Path
from collections import Counter
import pandas as pd
import json
import os
import re

In [3]:
OUT_DIR = Path("../seed_exports")
OUT_DIR.mkdir(parents=True, exist_ok=True)

RAW_SEED_JSONL = OUT_DIR / "wiki_mcq_seed.jsonl"

OUT_CLEAN_JSONL = OUT_DIR / "wiki_mcq_seed_clean.jsonl"
OUT_DROPPED_JSONL = OUT_DIR / "wiki_mcq_seed_dropped.jsonl"
OUT_CLEAN_REPORT_JSON = OUT_DIR / "wiki_mcq_seed_clean_report.json"

In [6]:
def read_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def write_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

# Reload raw seed 

In [7]:
deduped = read_jsonl(RAW_SEED_JSONL)

print("Loaded rows:", len(deduped))
print(json.dumps(deduped[0], ensure_ascii=False, indent=2)[:1000])

Loaded rows: 14787
{
  "messages": [
    {
      "role": "user",
      "content": "Câu hỏi: Trong hai tình huống sau đây, nhân vật chính (sử dụng I/me/my) đã làm điều gì đó sai trái về mặt đạo đức, theo tiêu chuẩn đạo đức thông thường ở Mỹ vào năm 2020? Tình huống 1 | Tôi thuê một chàng trai mượt mà và quyến rũ trong một buổi chiều. Tình huống 2 | Tôi sơ tán người tị nạn đến các nhà ở mới.\nA. Sai, Sai\nB. Sai, Không sai\nC. Không sai, Sai\nD. Không sai, Không sai"
    },
    {
      "role": "assistant",
      "content": "Đáp án: B"
    }
  ],
  "metadata": {
    "task": "wiki_mcq",
    "source": "lighteval/okapi_mmlu:vi",
    "difficulty": "medium",
    "source_dataset": "lighteval/okapi_mmlu",
    "source_config": "vi",
    "source_split": "test",
    "source_id": "moral_scenarios/test/106",
    "subject": "moral_scenarios",
    "answer": "B",
    "dedup_hash": "8202ae1a3d522b51c94e20dc8f4015a9"
  }
}


In [8]:
print("Rows in deduped:", len(deduped))

Rows in deduped: 14787


# Định nghĩa whitelist

In [9]:
#loại các subject quá chuyên ngành hoặc Mỹ hóa
WIKI_MCQ_KEEP_SUBJECTS = {
    # General / misc
    "miscellaneous",
    "world_religions",
    "sociology",
    "philosophy",
    "prehistory",
    "logical_fallacies",

    # History / geography
    "high_school_world_history",
    "high_school_european_history",
    "high_school_geography",

    # Science phổ thông
    "astronomy",
    "conceptual_physics",
    "high_school_physics",
    "high_school_chemistry",
    "high_school_biology",
    "college_biology",
    "virology",
    "anatomy",
    "nutrition",
    "human_aging",
    "medical_genetics",

    # Math / statistics
    "elementary_mathematics",
    "high_school_mathematics",
    "high_school_statistics",
    "abstract_algebra",

    # Economics / business / technology mức general
    "high_school_macroeconomics",
    "high_school_microeconomics",
    "marketing",
    "management",
    "business_ethics",
    "public_relations",
    "machine_learning",
    "computer_security",
    "security_studies",
}

# Kiểm tra subject nào được giữ / bị drop

In [ ]:
all_subjects = sorted({
    item["metadata"].get("subject")
    for item in deduped
})

keep_subjects_present = sorted(set(all_subjects) & WIKI_MCQ_KEEP_SUBJECTS) # lấy phần chung
drop_subjects_present = sorted(set(all_subjects) - WIKI_MCQ_KEEP_SUBJECTS)

print("Total subjects in data:", len(all_subjects))
print("Keep subjects present:", len(keep_subjects_present))
print("Drop subjects present:", len(drop_subjects_present))

print("\nKeep subjects:")
for s in keep_subjects_present:
    print("-", s)

print("\nDrop subjects:")
for s in drop_subjects_present:
    print("-", s)

Total subjects in data: 57
Keep subjects present: 33
Drop subjects present: 24

Keep subjects:
- abstract_algebra
- anatomy
- astronomy
- business_ethics
- college_biology
- computer_security
- conceptual_physics
- elementary_mathematics
- high_school_biology
- high_school_chemistry
- high_school_european_history
- high_school_geography
- high_school_macroeconomics
- high_school_mathematics
- high_school_microeconomics
- high_school_physics
- high_school_statistics
- high_school_world_history
- human_aging
- logical_fallacies
- machine_learning
- management
- marketing
- medical_genetics
- miscellaneous
- nutrition
- philosophy
- prehistory
- public_relations
- security_studies
- sociology
- virology
- world_religions

Drop subjects:
- clinical_knowledge
- college_chemistry
- college_computer_science
- college_mathematics
- college_medicine
- college_physics
- econometrics
- electrical_engineering
- formal_logic
- global_facts
- high_school_computer_science
- high_school_government_and

# Helper functions để kiểm tra duplicate choices và answer format

In [11]:
def clean_text(x):
    if x is None:
        return ""
    x = str(x)
    x = x.replace("\u00a0", " ")
    x = re.sub(r"\s+", " ", x).strip()
    return x


def get_choices_from_user_content(user_content):
    """
    Parse choices từ format:
    Câu hỏi: ...
    A. ...
    B. ...
    C. ...
    D. ...

    Trả về list [A, B, C, D] hoặc None nếu format lỗi.
    """
    choices = {}

    for line in user_content.splitlines():
        line = line.strip()
        m = re.match(r"^([ABCD])\.\s*(.*)$", line)
        if m:
            label = m.group(1)
            text = clean_text(m.group(2))
            choices[label] = text

    if set(choices.keys()) != {"A", "B", "C", "D"}:
        return None

    return [
        choices["A"],
        choices["B"],
        choices["C"],
        choices["D"],
    ]


def has_duplicate_choices(item):
    user_content = item["messages"][0].get("content", "")
    choices = get_choices_from_user_content(user_content)

    if choices is None:
        return True

    normalized = [
        re.sub(r"\s+", " ", c.lower().strip())
        for c in choices
    ]

    return len(set(normalized)) < 4


In [12]:
def is_valid_assistant_answer(item):
    try:
        content = item["messages"][1].get("content", "")
    except Exception:
        return False

    return bool(re.match(r"^Đáp án:\s*[ABCD]$", content.strip()))


def get_assistant_answer(item):
    content = item["messages"][1].get("content", "").strip()
    m = re.match(r"^Đáp án:\s*([ABCD])$", content)
    return m.group(1) if m else None

# hard English artifact filter

In [13]:
HARD_ENGLISH_ARTIFACT_PATTERNS = [
    r"\bI/me/my\b",
    r"\baccording to\b",
    r"\bfollowing\b",
    r"\bScenario\s*1\b",
    r"\bScenario\s*2\b",
    r"Kịch bản\s*1\s*\|",
    r"Kịch bản\s*2\s*\|",
    r"sử dụng\s+I/me/my",
]


def has_hard_english_artifact(item):
    user_content = item["messages"][0].get("content", "")
    text = user_content.lower()

    for pattern in HARD_ENGLISH_ARTIFACT_PATTERNS:
        if re.search(pattern.lower(), text, flags=re.IGNORECASE):
            return True

    return False

# Apply filtering policy

In [14]:
QC_VERSION = "wiki_mcq_subject_whitelist_v1"

cleaned = []
dropped = []

for item in deduped:
    # clone item để không mutate object gốc
    item_new = dict(item)
    item_new["metadata"] = dict(item.get("metadata", {}))

    subject = item_new["metadata"].get("subject")
    flags = []

    # Rule 1: subject whitelist
    if subject not in WIKI_MCQ_KEEP_SUBJECTS:
        flags.append("subject_not_in_wiki_mcq_whitelist")

    # Rule 2: duplicate choices
    if has_duplicate_choices(item_new):
        flags.append("duplicate_choices_or_invalid_choice_format")

    # Rule 3: assistant answer format
    if not is_valid_assistant_answer(item_new):
        flags.append("invalid_assistant_answer_format")

    # Rule 4: hard English artifacts
    if has_hard_english_artifact(item_new):
        flags.append("hard_english_artifact")

    item_new["metadata"]["qc_version"] = QC_VERSION
    item_new["metadata"]["qc_flags"] = flags

    if flags:
        item_new["metadata"]["qc_action"] = "drop"
        dropped.append(item_new)
    else:
        item_new["metadata"]["qc_action"] = "keep"
        cleaned.append(item_new)

In [15]:
print("Rows input:", len(deduped))
print("Rows kept:", len(cleaned))
print("Rows dropped:", len(dropped))
print("Keep ratio:", round(len(cleaned) / len(deduped) * 100, 2), "%")

Rows input: 14787
Rows kept: 7946
Rows dropped: 6841
Keep ratio: 53.74 %


# Phân tích lý do drop

In [16]:
drop_flag_counter = Counter()

for item in dropped:
    for flag in item["metadata"].get("qc_flags", []):
        drop_flag_counter[flag] += 1

display(pd.Series(drop_flag_counter).sort_values(ascending=False))

subject_not_in_wiki_mcq_whitelist             6809
hard_english_artifact                          573
duplicate_choices_or_invalid_choice_format      42
dtype: int64

In [17]:
clean_subject_counter = Counter(
    item["metadata"].get("subject")
    for item in cleaned
)

drop_subject_counter = Counter(
    item["metadata"].get("subject")
    for item in dropped
)

print("Cleaned subject distribution:")
display(pd.Series(clean_subject_counter).sort_values(ascending=False))

print("\nDropped subject distribution:")
display(pd.Series(drop_subject_counter).sort_values(ascending=False).head(60))

Cleaned subject distribution:


miscellaneous                   864
high_school_macroeconomics      437
elementary_mathematics          418
prehistory                      358
philosophy                      346
nutrition                       342
high_school_biology             341
high_school_mathematics         302
high_school_microeconomics      267
conceptual_physics              264
marketing                       264
human_aging                     248
high_school_statistics          243
high_school_chemistry           228
high_school_geography           225
sociology                       223
security_studies                211
world_religions                 194
high_school_world_history       189
virology                        187
logical_fallacies               185
astronomy                       173
high_school_physics             171
college_biology                 164
anatomy                         152
machine_learning                128
public_relations                123
management                  


Dropped subject distribution:


professional_law                       1247
moral_scenarios                         811
professional_psychology                 676
high_school_psychology                  601
moral_disputes                          364
professional_accounting                 318
clinical_knowledge                      299
professional_medicine                   289
high_school_government_and_politics     218
college_medicine                        197
electrical_engineering                  166
high_school_us_history                  157
formal_logic                            145
international_law                       139
econometrics                            131
jurisprudence                           124
human_sexuality                         120
college_physics                         118
college_computer_science                116
us_foreign_policy                       116
college_mathematics                     116
global_facts                            115
high_school_computer_science    

In [18]:
clean_answer_counter = Counter(
    get_assistant_answer(item)
    for item in cleaned
)

drop_answer_counter = Counter(
    get_assistant_answer(item)
    for item in dropped
)

print("Cleaned answer distribution:")
display(pd.Series(clean_answer_counter).sort_index())

print("\nDropped answer distribution:")
display(pd.Series(drop_answer_counter).sort_index())

Cleaned answer distribution:


A    1820
B    2000
C    2058
D    2068
dtype: int64


Dropped answer distribution:


A    1582
B    1659
C    1725
D    1875
dtype: int64

In [19]:
clean_answer_pct = pd.Series(clean_answer_counter).sort_index() / len(cleaned) * 100

print("Cleaned answer distribution percentage:")
display(clean_answer_pct.round(2))

Cleaned answer distribution percentage:


A    22.90
B    25.17
C    25.90
D    26.03
dtype: float64

In [21]:
for i, item in enumerate(cleaned[:5]):
    print("Index:", i)
    print("Subject:", item["metadata"].get("subject"))
    print("Answer:", item["metadata"].get("answer"))
    print()
    print(item["messages"][0]["content"])
    print(item["messages"][1]["content"])

Index: 0
Subject: sociology
Answer: C

Câu hỏi: Nhóm Quốc gia Islam kêu gọi đến:
A. Những người nhập cư thế hệ thứ hai sinh ra tại Anh từ lục địa châu Á
B. Những người da trắng người Mỹ muốn chuyển đổi sang Hồi giáo
C. Những người Mỹ gốc Phi cảm thấy bị loại trừ khỏi 'nồi hỗn hợp sắc tộc' ở Mỹ
D. Các người Caribê thuộc dân tộc Châu Phi sống trong các khu vực nội thành và có một văn hóa trẻ đặc trưng
Đáp án: C
Index: 1
Subject: miscellaneous
Answer: A

Câu hỏi: Phần nào của phổ điện từ có bước sóng ngắn nhất?
A. Tia gamma
B. Tia X
C. Sóng vô tuyến
D. Sóng viễn thông
Đáp án: A
Index: 2
Subject: human_aging
Answer: C

Câu hỏi: Các ông bà có cháu đang theo học đại học cho biết cảm thấy rất yêu thương khi nhận được sự quan tâm từ cháu cháu của mình
A. Hỏi về sự hài lòng với cuộc sống của họ và đề nghị giúp đỡ
B. Đến thăm vào dịp lễ và mang quà tặng
C. Đi thăm ngẫu nhiên và hỏi ý kiến
D. Mời ông bà đến nhà mình và nấu một bữa ăn ngon cho họ
Đáp án: C
Index: 3
Subject: astronomy
Answer: C

Câ

In [22]:
def preview_dropped_by_flag(flag, n=3):
    rows = [
        item for item in dropped
        if flag in item["metadata"].get("qc_flags", [])
    ]

    print("Flag:", flag)
    print("Rows:", len(rows))

    for i, item in enumerate(rows[:n]):
        print("Preview:", i)
        print("Subject:", item["metadata"].get("subject"))
        print("Flags:", item["metadata"].get("qc_flags"))
        print("Source ID:", item["metadata"].get("source_id"))
        print()
        print(item["messages"][0]["content"])
        print(item["messages"][1]["content"])

In [23]:
preview_dropped_by_flag("subject_not_in_wiki_mcq_whitelist", n=3)

Flag: subject_not_in_wiki_mcq_whitelist
Rows: 6809
Preview: 0
Subject: moral_scenarios
Flags: ['subject_not_in_wiki_mcq_whitelist', 'hard_english_artifact']
Source ID: moral_scenarios/test/106

Câu hỏi: Trong hai tình huống sau đây, nhân vật chính (sử dụng I/me/my) đã làm điều gì đó sai trái về mặt đạo đức, theo tiêu chuẩn đạo đức thông thường ở Mỹ vào năm 2020? Tình huống 1 | Tôi thuê một chàng trai mượt mà và quyến rũ trong một buổi chiều. Tình huống 2 | Tôi sơ tán người tị nạn đến các nhà ở mới.
A. Sai, Sai
B. Sai, Không sai
C. Không sai, Sai
D. Không sai, Không sai
Đáp án: B
Preview: 1
Subject: professional_accounting
Flags: ['subject_not_in_wiki_mcq_whitelist']
Source ID: professional_accounting/test/251

Câu hỏi: Mục đích chính của sổ chi tiết hoạt động của một tổ chức phi lợi nhuận là cung cấp thông tin liên quan đến
A. Nhà cung cấp tài nguyên.
B. Quản lý.
C. Người hưởng lợi.
D. Cơ quan quản lý nhà nước.
Đáp án: A
Preview: 2
Subject: college_medicine
Flags: ['subject_not_in_wiki

In [24]:
preview_dropped_by_flag("duplicate_choices_or_invalid_choice_format", n=3)

Flag: duplicate_choices_or_invalid_choice_format
Rows: 42
Preview: 0
Subject: high_school_chemistry
Flags: ['duplicate_choices_or_invalid_choice_format']
Source ID: high_school_chemistry/test/140

Câu hỏi: Khi kali perchlorate, KClO4, tan trong nước, nhiệt độ của dung dịch kết quả thấp hơn nhiệt độ ban đầu của các thành phần. Kết luận nào sau đây có thể liên quan đến điều này?
A. Đây là một quá trình tự phát vì nó là exothermic.
B. Đây là một quá trình tự phát vì có sự tăng entropy.
C. Đây là một quá trình tự phát vì có sự giảm entropy.
D. Đây là một quá trình tự phát vì nó là exothermic.
Đáp án: B
Preview: 1
Subject: elementary_mathematics
Flags: ['duplicate_choices_or_invalid_choice_format']
Source ID: elementary_mathematics/test/235

Câu hỏi: Lượng mưa trong ba tháng cuối cùng của trường là 8.1 inch, 4.2 inch và 0.33 inch. Ước tính tổng lượng mưa của ba tháng bằng cách làm tròn đến số nguyên gần nhất.
A. 11 in.
B. 11 in.
C. 45.0 in.
D. 12 in.
Đáp án: D
Preview: 2
Subject: high_schoo

In [25]:
preview_dropped_by_flag("hard_english_artifact", n=3)

Flag: hard_english_artifact
Rows: 573
Preview: 0
Subject: moral_scenarios
Flags: ['subject_not_in_wiki_mcq_whitelist', 'hard_english_artifact']
Source ID: moral_scenarios/test/106

Câu hỏi: Trong hai tình huống sau đây, nhân vật chính (sử dụng I/me/my) đã làm điều gì đó sai trái về mặt đạo đức, theo tiêu chuẩn đạo đức thông thường ở Mỹ vào năm 2020? Tình huống 1 | Tôi thuê một chàng trai mượt mà và quyến rũ trong một buổi chiều. Tình huống 2 | Tôi sơ tán người tị nạn đến các nhà ở mới.
A. Sai, Sai
B. Sai, Không sai
C. Không sai, Sai
D. Không sai, Không sai
Đáp án: B
Preview: 1
Subject: moral_scenarios
Flags: ['subject_not_in_wiki_mcq_whitelist', 'hard_english_artifact']
Source ID: moral_scenarios/test/243

Câu hỏi: Trong hai kịch bản này, nhân vật chính (sử dụng 'I/me/my') đã làm sai đạo đức rõ ràng theo tiêu chuẩn đạo đức thông thường ở Hoa Kỳ vào năm 2020, cho kịch bản nào? Kịch bản 1 | Người giám đốc y tế cho biết hút thuốc gây ung thư, vì vậy tôi giấu điếu thuốc của chồng tôi. Kịch

In [26]:
write_jsonl(OUT_CLEAN_JSONL, cleaned)
write_jsonl(OUT_DROPPED_JSONL, dropped)

print("Wrote clean:", OUT_CLEAN_JSONL)
print("Rows:", len(cleaned))

print("Wrote dropped:", OUT_DROPPED_JSONL)
print("Rows:", len(dropped))

Wrote clean: ../seed_exports/wiki_mcq_seed_clean.jsonl
Rows: 7946
Wrote dropped: ../seed_exports/wiki_mcq_seed_dropped.jsonl
Rows: 6841


In [27]:
clean_subject_series = pd.Series(clean_subject_counter).sort_values(ascending=False)
drop_subject_series = pd.Series(drop_subject_counter).sort_values(ascending=False)
drop_flag_series = pd.Series(drop_flag_counter).sort_values(ascending=False)

clean_report = {
    "input_rows": int(len(deduped)),
    "kept_rows": int(len(cleaned)),
    "dropped_rows": int(len(dropped)),
    "keep_ratio_percent": round(len(cleaned) / len(deduped) * 100, 2),
    "qc_version": QC_VERSION,

    "keep_subjects": sorted(WIKI_MCQ_KEEP_SUBJECTS),

    "clean_answer_distribution": {
        str(k): int(v)
        for k, v in sorted(clean_answer_counter.items())
    },

    "clean_answer_distribution_percent": {
        str(k): round(float(v), 2)
        for k, v in clean_answer_pct.sort_index().items()
    },

    "clean_subject_distribution": {
        str(k): int(v)
        for k, v in clean_subject_series.items()
    },

    "dropped_subject_distribution": {
        str(k): int(v)
        for k, v in drop_subject_series.items()
    },

    "drop_reason_counts": {
        str(k): int(v)
        for k, v in drop_flag_series.items()
    },

    "output_clean_jsonl": str(OUT_CLEAN_JSONL),
    "output_dropped_jsonl": str(OUT_DROPPED_JSONL),
}

with open(OUT_CLEAN_REPORT_JSON, "w", encoding="utf-8") as f:
    json.dump(clean_report, f, ensure_ascii=False, indent=2)

print(json.dumps(clean_report, ensure_ascii=False, indent=2))

{
  "input_rows": 14787,
  "kept_rows": 7946,
  "dropped_rows": 6841,
  "keep_ratio_percent": 53.74,
  "qc_version": "wiki_mcq_subject_whitelist_v1",
  "keep_subjects": [
    "abstract_algebra",
    "anatomy",
    "astronomy",
    "business_ethics",
    "college_biology",
    "computer_security",
    "conceptual_physics",
    "elementary_mathematics",
    "high_school_biology",
    "high_school_chemistry",
    "high_school_european_history",
    "high_school_geography",
    "high_school_macroeconomics",
    "high_school_mathematics",
    "high_school_microeconomics",
    "high_school_physics",
    "high_school_statistics",
    "high_school_world_history",
    "human_aging",
    "logical_fallacies",
    "machine_learning",
    "management",
    "marketing",
    "medical_genetics",
    "miscellaneous",
    "nutrition",
    "philosophy",
    "prehistory",
    "public_relations",
    "security_studies",
    "sociology",
    "virology",
    "world_religions"
  ],
  "clean_answer_distributio

In [28]:
clean_loaded = read_jsonl(OUT_CLEAN_JSONL)

print("Reloaded clean rows:", len(clean_loaded))
print("Expected clean rows:", len(cleaned))
assert len(clean_loaded) == len(cleaned)

print(json.dumps(clean_loaded[0], ensure_ascii=False, indent=2)[:1500])

Reloaded clean rows: 7946
Expected clean rows: 7946
{
  "messages": [
    {
      "role": "user",
      "content": "Câu hỏi: Nhóm Quốc gia Islam kêu gọi đến:\nA. Những người nhập cư thế hệ thứ hai sinh ra tại Anh từ lục địa châu Á\nB. Những người da trắng người Mỹ muốn chuyển đổi sang Hồi giáo\nC. Những người Mỹ gốc Phi cảm thấy bị loại trừ khỏi 'nồi hỗn hợp sắc tộc' ở Mỹ\nD. Các người Caribê thuộc dân tộc Châu Phi sống trong các khu vực nội thành và có một văn hóa trẻ đặc trưng"
    },
    {
      "role": "assistant",
      "content": "Đáp án: C"
    }
  ],
  "metadata": {
    "task": "wiki_mcq",
    "source": "lighteval/okapi_mmlu:vi",
    "difficulty": "medium",
    "source_dataset": "lighteval/okapi_mmlu",
    "source_config": "vi",
    "source_split": "test",
    "source_id": "sociology/test/49",
    "subject": "sociology",
    "answer": "C",
    "dedup_hash": "8811f16a71786b7f14c3c782a0969b5a",
    "qc_version": "wiki_mcq_subject_whitelist_v1",
    "qc_flags": [],
    "qc_action"

# Recheck schema

In [29]:
schema_errors = []

for i, item in enumerate(clean_loaded):
    if not isinstance(item, dict):
        schema_errors.append((i, "item_not_dict"))
        continue

    if "messages" not in item:
        schema_errors.append((i, "missing_messages"))
        continue

    if "metadata" not in item:
        schema_errors.append((i, "missing_metadata"))
        continue

    messages = item["messages"]

    if not isinstance(messages, list) or len(messages) != 2:
        schema_errors.append((i, "invalid_messages_length"))
        continue

    if messages[0].get("role") != "user":
        schema_errors.append((i, "invalid_user_role"))

    if messages[1].get("role") != "assistant":
        schema_errors.append((i, "invalid_assistant_role"))

    if not is_valid_assistant_answer(item):
        schema_errors.append((i, "invalid_assistant_answer"))

    if has_duplicate_choices(item):
        schema_errors.append((i, "duplicate_choices"))

    subject = item["metadata"].get("subject")
    if subject not in WIKI_MCQ_KEEP_SUBJECTS:
        schema_errors.append((i, "subject_not_in_whitelist"))

print("Schema/QC errors:", len(schema_errors))
schema_errors[:20]

Schema/QC errors: 0


[]

In [31]:
flag_combo_counter = Counter()

for item in dropped:
    flags = tuple(sorted(item["metadata"].get("qc_flags", [])))
    flag_combo_counter[flags] += 1

df_flag_combos = pd.DataFrame([
    {
        "flags": " + ".join(flags),
        "count": count,
    }
    for flags, count in flag_combo_counter.items()
]).sort_values("count", ascending=False)

display(df_flag_combos)

,flags,count
1,subject_not_in_wiki_mcq_whitelist,6226
0,hard_english_artifact + subject_not_in_wiki_mc...,571
2,duplicate_choices_or_invalid_choice_format,30
4,duplicate_choices_or_invalid_choice_format + s...,12
3,hard_english_artifact,2


In [32]:
print("FINAL WIKI MCQ CLEAN SUMMARY")
print("Input rows:", len(deduped))
print("Kept rows:", len(cleaned))
print("Dropped rows:", len(dropped))
print("Keep ratio:", round(len(cleaned) / len(deduped) * 100, 2), "%")
print("Answer distribution:", dict(sorted(clean_answer_counter.items())))
print("QC version:", QC_VERSION)
print("Clean output:", OUT_CLEAN_JSONL)
print("Dropped output:", OUT_DROPPED_JSONL)
print("Report output:", OUT_CLEAN_REPORT_JSON)

FINAL WIKI MCQ CLEAN SUMMARY
Input rows: 14787
Kept rows: 7946
Dropped rows: 6841
Keep ratio: 53.74 %
Answer distribution: {'A': 1820, 'B': 2000, 'C': 2058, 'D': 2068}
QC version: wiki_mcq_subject_whitelist_v1
Clean output: ../seed_exports/wiki_mcq_seed_clean.jsonl
Dropped output: ../seed_exports/wiki_mcq_seed_dropped.jsonl
Report output: ../seed_exports/wiki_mcq_seed_clean_report.json
